In [ ]:
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import json
from ast import literal_eval

import requests

from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer

import numpy as np
import sklearn.metrics.pairwise as pw

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import scipy

import math
import random
import sklearn
from nltk.corpus import stopwords
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split

from scipy.sparse.linalg import svds
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [ ]:
import os
from langchain_community.llms import Ollama
from langchain.embeddings import OllamaEmbeddings

ollama_url = os.getenv('OLLAMA_URL')
model = OllamaEmbeddings(base_url=ollama_url, model="nomic-embed-text")

with open('./data/youtube_history.json', 'r', encoding='utf-8') as f:
	videos = json.load(f)

for video in videos:
	text = video['title']
	if 'tags' in video:
		text += f" {" ".join(video['tags'])}"
	
	video['soup'] = text
	video['embeddings'] = model.embed_query(text)
	
with open('./data/youtube_history_embeddings.json', 'w', encoding='utf-8') as f:
  json.dump(videos, f, ensure_ascii=False, indent=2)
	 

In [ ]:
videos = pd.read_json('./data/youtube_history_embeddings.json') 
#print(videos)
print(videos.info()) 


#print(pd.Series(videos.index, index=videos['title']).drop_duplicates())

# metadata = videos.reset_index()
# #print(metadata)
# indices = pd.Series(metadata.index, index=metadata['title']).drop_duplicates()
# print(indices)

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import List

@dataclass
class Article:
    feed_name: str
    title: str
    link: str
    summary: str
    pub_date: datetime
    updated: datetime
    distance: float = None

    def __init__(self, feed_name: str, title: str, link: str, summary: str, pub_date: datetime, updated: datetime):
        self.feed_name = feed_name
        self.title = title
        self.link = link
        self.summary = summary
        self.pub_date = pub_date
        self.updated = updated

In [ ]:
import html
from bs4 import BeautifulSoup
import feedparser

def clean_text(text):
  text = text.replace('\n', ' ').replace('\r', ' ').strip()
  text = BeautifulSoup(html.unescape(text), 'lxml').text
  return text

def parse_feed(url):
	feed_item = feedparser.parse(url)
	entries = []
	for article in feed_item.entries:
		entries.append(Article(
			feed_name = feed_item.feed.title,
			title = article.get('title', ''),
			link = article.link,
			summary = article.get('summary', ''),
			pub_date = datetime.now(),
			updated = datetime.now()
		))
  
	return entries

entries =  parse_feed('https://www.reddit.com/r/selfhosted/rising.rss')
entries += parse_feed("https://www.battleswarmblog.com/?feed=rss2")
entries += parse_feed("https://cafehayek.com/feed")
entries += parse_feed("https://rss.slashdot.org/Slashdot/slashdotMain")
entries += parse_feed("https://www.realclearpolitics.com/index.xml")
entries += parse_feed("https://instapundit.com/feed/")
entries += parse_feed("https://twitchy.com/feed")
entries += parse_feed("https://hnrss.org/frontpage")
entries += parse_feed("https://www.reddit.com/r/selfhosted/top.rss?t=day")

In [ ]:
import json
from dataclasses import asdict
with open('./data/feed_entries.json', 'w', encoding='utf-8') as f:
  json.dump([{**asdict(p), "pub_date": p.pub_date.isoformat(), "updated": p.updated.isoformat()} for p in entries], f, ensure_ascii=False, indent=2)